# Bank Customer Churn KDD — Phase 3

## Association Rule Mining

**Owner:** Pattern Analyst  
**Decision standard:** expose the complete rule funnel, retain high-support/high-confidence/high-lift churn associations, remove extensions that add negligible confidence over a simpler parent, and deliver at least 10 rules with a specific operational interpretation.

Rules describe co-occurrence in this snapshot. They are not prediction claims and do not establish causality.

In [1]:
# Shared imports and project-root-aware artifact paths
from pathlib import Path
import sys
import warnings

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, silhouette_samples
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import cdist
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
from sklearn.ensemble import IsolationForest
from scipy import stats
from IPython.display import display, Markdown

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 5)})

PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (
            (candidate / 'src' / '_pipeline_utils.py').is_file()
            and (candidate / 'data' / 'raw' / 'churn.csv').is_file()
        )
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        'Open this notebook from inside the Project_DATAMINING repository.'
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src._pipeline_utils import upsert_evaluation_metrics

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'churn.csv'
CLEAN_PATH = PROCESSED_DIR / 'churn_clean.csv'
CLUSTER_MATRIX = PROCESSED_DIR / 'churn_clustering_matrix.csv'
TRANSACTIONS_PATH = PROCESSED_DIR / 'churn_transactions.csv'
OHE_TRANSACTIONS_PATH = PROCESSED_DIR / 'churn_ohe_transactions.csv'
CLUSTERED_PATH = PROCESSED_DIR / 'churn_clustered.csv'
DBSCAN_OUTLIERS_PATH = PROCESSED_DIR / 'dbscan_outlier_indices.npy'
TOP_RULES_PATH = OUTPUTS_DIR / 'ph3_top_association_rules.csv'
ALL_RULES_PATH = OUTPUTS_DIR / 'ph3_all_association_rules.csv'
ANOMALY_REPORT_PATH = OUTPUTS_DIR / 'ph4_anomaly_report.csv'
HIGH_CONFIDENCE_ANOMALIES_PATH = OUTPUTS_DIR / 'ph4_high_confidence_anomalies.csv'

def require_artifacts(paths, phase_name):
    missing = [path for path in paths if not Path(path).is_file()]
    if missing:
        formatted = '\n'.join(f'  - {path}' for path in missing)
        raise FileNotFoundError(
            f'{phase_name} is missing prerequisite artifacts:\n{formatted}'
        )

print('✔ All libraries loaded successfully.')
print(f'  Project root: {PROJECT_ROOT}')
print(f'  Python:       {sys.version.split()[0]}')
print(f'  Executable:   {sys.executable}')
print(f'  Pandas:       {pd.__version__}')
print(f'  NumPy:        {np.__version__}')


✔ All libraries loaded successfully.
  Project root: C:\Users\Thomas\Documents\Project_DATAMINING
  Python:       3.11.9
  Executable:   C:\Users\Thomas\Documents\Project_DATAMINING\.venv\Scripts\python.exe
  Pandas:       2.2.2
  NumPy:        1.26.4


## Prerequisites

Requires Phase 1's clean dataset and one-hot transaction matrix. Phase 2 is not required.

In [2]:
require_artifacts(
    [CLEAN_PATH, OHE_TRANSACTIONS_PATH],
    'Phase 3',
)
df = pd.read_csv(CLEAN_PATH)
_ohe_row_count = len(pd.read_csv(OHE_TRANSACTIONS_PATH, usecols=[0]))
if len(df) != _ohe_row_count:
    raise ValueError('Phase 3 clean data and transaction row counts differ. Rerun Phase 1.')
print(f'✔ Phase 3 prerequisites loaded: {len(df):,} aligned transactions.')


✔ Phase 3 prerequisites loaded: 10,000 aligned transactions.


## Phase 3: Association Rule Mining (Apriori Algorithm)

**Objective:** Discover non-obvious co-occurrence patterns among customer behavioral and financial attributes, with focus on identifying strong churn signals.

**Key Hypothesis to Test:**  
> "Customers from Germany holding only one product who are inactive represent a strong churn profile."

**Algorithm:** Apriori (mlxtend)  
**Dataset:** Path B transaction matrix (`churn_ohe_transactions.csv`)  
**Filter Criteria:** Minimum support = 0.03, confidence >= 0.50, lift >= 1.5  
**Rationale:** A 0.05 support floor produced too few churn-consequent rules for the rubric. The 0.03 floor still represents about 300 customers in this 10,000-record dataset and produced 17 rules whose consequent includes churn: 16 have the exact single-item consequent `{Churned}`, while one has a multi-item consequent that also includes churn. The exact-single subset supplies 11 non-redundant rules, of which the top 10 form the documented deliverable.
**Balance items:** zero, 0–100K, and above-100K scenario bands. The source does not document currency or insurance status; see Phase 1 for the rationale and sensitivity limitation.  
**Deliverable:** At least 10 non-trivial, high-lift association rules with business interpretation


In [3]:
# ── Load OHE Transaction Matrix ───────────────────────────────────
df_ohe_txn = pd.read_csv(OHE_TRANSACTIONS_PATH)

# Ensure boolean dtype
df_ohe_txn = df_ohe_txn.astype(bool)

# Drop the negation of the target. Keeping 'Churn_Status_Retained' as a
# minable item produces tautological rules of the form {Retained,...} -> {X}
# that carry no business insight. Phase 3 mines rules whose CONSEQUENT is
# Churned; the Retained column is its complement and adds only noise.
if 'Churn_Status_Retained' in df_ohe_txn.columns:
    df_ohe_txn = df_ohe_txn.drop(columns=['Churn_Status_Retained'])

print("── Transaction Matrix Audit ──")
print(f"  Shape: {df_ohe_txn.shape}")
print(f"  Transactions (rows): {len(df_ohe_txn):,}")
print(f"  Items (columns): {df_ohe_txn.shape[1]}")
print(f"  Density: {df_ohe_txn.values.mean()*100:.2f}% (avg items per row / total items)")
print(f"\n  Columns Preview:")
for col in sorted(df_ohe_txn.columns):
    support = df_ohe_txn[col].mean()
    print(f"    {col:<40} support = {support:.4f} ({support*100:.1f}%)")


── Transaction Matrix Audit ──
  Shape: (10000, 33)
  Transactions (rows): 10,000
  Items (columns): 33
  Density: 30.92% (avg items per row / total items)

  Columns Preview:
    Active_Status_Active                     support = 0.5151 (51.5%)
    Active_Status_Inactive                   support = 0.4849 (48.5%)
    Age_Band_Age_18_to_30                    support = 0.1968 (19.7%)
    Age_Band_Age_31_to_45                    support = 0.5921 (59.2%)
    Age_Band_Age_46_to_60                    support = 0.1647 (16.5%)
    Age_Band_Age_61_plus                     support = 0.0464 (4.6%)
    Balance_Band_Balance_0_to_100K           support = 0.1584 (15.8%)
    Balance_Band_Balance_Above_100K          support = 0.4799 (48.0%)
    Balance_Band_Zero_Balance                support = 0.3617 (36.2%)
    Churn_Status_Churned                     support = 0.2037 (20.4%)
    CrCard_Status_Has_CrCard                 support = 0.7055 (70.5%)
    CrCard_Status_No_CrCard                  support = 

In [4]:
# ── Apriori: Frequent Itemset Mining ──────────────────────────────────
# Churn base-rate ≈ 20%. A min_support of 0.05 demanded the antecedent+
# Churned itemset appear in >=5% of customers, which only 3 multi-attribute
# churn rules cleared. PDF Phase 3 rubric requires >=10 non-trivial rules,
# so we lower the support floor to 0.03 (≈300 customers — still a defensible
# segment size, not a single-record fluke).
MIN_SUPPORT    = 0.03   # Item combination must appear in >=3% of customers
MAX_LEN        = 5      # Maximum itemset length (controls complexity)

print(f"Running Apriori (min_support={MIN_SUPPORT}, max_len={MAX_LEN})...")
frequent_itemsets = apriori(
    df_ohe_txn,
    min_support=MIN_SUPPORT,
    use_colnames=True,
    max_len=MAX_LEN,
    verbose=1
)

frequent_itemsets['itemset_size'] = frequent_itemsets['itemsets'].apply(len)
frequent_itemsets = frequent_itemsets.sort_values('support', ascending=False)

print(f"\n── Frequent Itemset Summary ──")
print(f"  Total frequent itemsets found: {len(frequent_itemsets):,}")
print(f"\n  By itemset size:")
display(frequent_itemsets['itemset_size'].value_counts().sort_index()
          .rename('Count').to_frame())

print(f"\n── Top 20 Most Frequent Itemsets ──")
display(frequent_itemsets.head(20)[['support','itemset_size','itemsets']]
          .reset_index(drop=True))


Running Apriori (min_support=0.03, max_len=5)...

Processing 930 combinations | Sampling itemset size 2
Processing 9915 combinations | Sampling itemset size 3
Processing 33892 combinations | Sampling itemset size 4


Processing 30025 combinations | Sampling itemset size 5



── Frequent Itemset Summary ──
  Total frequent itemsets found: 4,105

  By itemset size:


,Count
itemset_size,
1,31
2,380
3,1588
4,1728
5,378



── Top 20 Most Frequent Itemsets ──


,support,itemset_size,itemsets
0,0.7055,1,(CrCard_Status_Has_CrCard)
1,0.5921,1,(Age_Band_Age_31_to_45)
2,0.5457,1,(Gender_Male)
3,0.5151,1,(Active_Status_Active)
4,0.5084,1,(Products_Label_Products_1)
5,0.5014,1,(Geography_France)
6,0.4849,1,(Active_Status_Inactive)
7,0.4799,1,(Balance_Band_Balance_Above_100K)
8,0.4590,1,(Products_Label_Products_2)
9,0.4543,1,(Gender_Female)


### Interpretation — Frequent ≠ Interesting

The top of the frequency table is deliberately unexciting, and understanding why matters:

- **The most frequent "patterns" are single high-base-rate items** (has a credit card 70.6%, middle-aged 59.2%, male 54.6%) **and their pairwise products.** The top-2 itemset {Has_CrCard, Middle_Aged} at support 0.4176 is almost exactly 0.7055 × 0.5921 = 0.4177 — pure statistical independence (lift ≈ 1.00). High support here is an arithmetic consequence of marginal frequencies, not co-occurrence knowledge.
- **4,105 frequent itemsets from 33 items** shows the combinatorial scale even at min_support = 0.03. Sizes 3–4 dominate (3,316 itemsets) because every customer contributes one item per attribute family (10–11 of the 33 items — hence the 31% matrix density), so mid-size combinations are mechanically abundant.
- **This is exactly why the next cell filters on confidence ≥ 0.50 *and* lift ≥ 1.5:** confidence imposes decision-usefulness ("given the antecedent, churn is more likely than not"), and lift removes base-rate artifacts like the itemsets above by demanding a ≥50% deviation from independence.

**Conclusion:** frequency finds the haystack; interestingness measures find the needles. Reporting raw frequent itemsets as "findings" would be the classic ARM mistake this filtering pipeline is designed to avoid.


In [5]:
# ── Rule generation and transparent filtering funnel ───────────────────
MIN_CONFIDENCE = 0.50
MIN_LIFT = 1.50
MIN_INCREMENTAL_CONFIDENCE = 0.01

# Generate the complete rule universe from the supported itemsets first; each
# later filter is counted separately so "generated" and "retained" are not
# conflated in the appendix.
rules_raw = association_rules(
    frequent_itemsets,
    metric='confidence',
    min_threshold=0.0,
)
rules_confidence = rules_raw[rules_raw['confidence'] >= MIN_CONFIDENCE].copy()
rules = rules_confidence[rules_confidence['lift'] >= MIN_LIFT].copy()

for frame in (rules_raw, rules_confidence, rules):
    frame['antecedent_len'] = frame['antecedents'].apply(len)
    frame['consequent_len'] = frame['consequents'].apply(len)

rules['conviction'] = (
    (1 - rules['consequent support']) / (1 - rules['confidence'] + 1e-9)
)
rules = rules.sort_values('lift', ascending=False).reset_index(drop=True)

def excludes_retained_and_tautology(frame):
    return frame[
        ~frame['antecedents'].apply(lambda x: 'Churn_Status_Retained' in x)
        & ~frame['consequents'].apply(lambda x: 'Churn_Status_Retained' in x)
        & ~frame['antecedents'].apply(lambda x: 'Churn_Status_Churned' in x)
    ].copy()

rules = excludes_retained_and_tautology(rules)
churn_rules = rules[
    rules['consequents'].apply(lambda x: 'Churn_Status_Churned' in x)
].copy()
churn_single_rules = churn_rules[
    churn_rules['consequents'] == frozenset({'Churn_Status_Churned'})
].copy()
non_churn_rules = rules[
    ~rules['consequents'].apply(lambda x: 'Churn_Status_Churned' in x)
].copy()

# Compare each candidate with the strongest proper-subset churn rule from the
# full supported rule universe. Card/balance extensions that do not improve
# confidence are therefore excluded from the top-10 deliverable.
raw_churn_single = excludes_retained_and_tautology(rules_raw)
raw_churn_single = raw_churn_single[
    raw_churn_single['consequents'] == frozenset({'Churn_Status_Churned'})
].copy()

def best_parent_confidence(antecedent):
    parent_rows = raw_churn_single[
        raw_churn_single['antecedents'].apply(lambda parent: parent < antecedent)
    ]
    return float(parent_rows['confidence'].max()) if len(parent_rows) else np.nan

churn_single_rules['Best_Parent_Confidence'] = churn_single_rules['antecedents'].apply(
    best_parent_confidence
)
churn_single_rules['Incremental_Confidence'] = (
    churn_single_rules['confidence'] - churn_single_rules['Best_Parent_Confidence']
)
nonredundant_churn_rules = churn_single_rules[
    churn_single_rules['Best_Parent_Confidence'].isna()
    | (churn_single_rules['Incremental_Confidence'] >= MIN_INCREMENTAL_CONFIDENCE)
].sort_values('lift', ascending=False).reset_index(drop=True)

# MAX_LEN sensitivity: a retained five-item rule reaches the primary search
# cap, so rerun the search one level deeper and compare the actual deliverable.
SENSITIVITY_MAX_LEN = MAX_LEN + 1
sensitivity_itemsets = apriori(
    df_ohe_txn,
    min_support=MIN_SUPPORT,
    use_colnames=True,
    max_len=SENSITIVITY_MAX_LEN,
    verbose=0,
)
sensitivity_rules_raw = association_rules(
    sensitivity_itemsets, metric='confidence', min_threshold=0.0
)
sensitivity_rules = sensitivity_rules_raw[
    (sensitivity_rules_raw['confidence'] >= MIN_CONFIDENCE)
    & (sensitivity_rules_raw['lift'] >= MIN_LIFT)
].copy()
sensitivity_rules = excludes_retained_and_tautology(sensitivity_rules)
sensitivity_churn_single = sensitivity_rules[
    sensitivity_rules['consequents'] == frozenset({'Churn_Status_Churned'})
].copy()
sensitivity_raw_churn_single = excludes_retained_and_tautology(
    sensitivity_rules_raw
)
sensitivity_raw_churn_single = sensitivity_raw_churn_single[
    sensitivity_raw_churn_single['consequents']
    == frozenset({'Churn_Status_Churned'})
].copy()

def sensitivity_best_parent_confidence(antecedent):
    parent_rows = sensitivity_raw_churn_single[
        sensitivity_raw_churn_single['antecedents'].apply(
            lambda parent: parent < antecedent
        )
    ]
    return float(parent_rows['confidence'].max()) if len(parent_rows) else np.nan

sensitivity_churn_single['Best_Parent_Confidence'] = (
    sensitivity_churn_single['antecedents'].apply(
        sensitivity_best_parent_confidence
    )
)
sensitivity_churn_single['Incremental_Confidence'] = (
    sensitivity_churn_single['confidence']
    - sensitivity_churn_single['Best_Parent_Confidence']
)
sensitivity_nonredundant = sensitivity_churn_single[
    sensitivity_churn_single['Best_Parent_Confidence'].isna()
    | (sensitivity_churn_single['Incremental_Confidence']
       >= MIN_INCREMENTAL_CONFIDENCE)
].copy()

def rule_signatures(frame):
    return {
        (tuple(sorted(antecedent)), tuple(sorted(consequent)))
        for antecedent, consequent in zip(frame['antecedents'], frame['consequents'])
    }

primary_signatures = rule_signatures(nonredundant_churn_rules)
sensitivity_signatures = rule_signatures(sensitivity_nonredundant)
max_retained_len = int(
    nonredundant_churn_rules['antecedents'].apply(len).max()
) + 1
sensitivity_max_retained_len = int(
    sensitivity_nonredundant['antecedents'].apply(len).max()
) + 1
print(f"  MAX_LEN sensitivity: primary retained maximum={max_retained_len} "
      f"items at cap {MAX_LEN}; expanded search cap={SENSITIVITY_MAX_LEN}.")
print(f"  Expanded search adds {len(sensitivity_itemsets) - len(frequent_itemsets):,} "
      f"frequent itemsets and "
      f"{len(sensitivity_rules_raw) - len(rules_raw):,} raw rules.")
print(f"  Retained non-redundant churn rules: "
      f"{len(nonredundant_churn_rules)} -> {len(sensitivity_nonredundant)}; "
      f"expanded retained maximum={sensitivity_max_retained_len} items.")
assert primary_signatures == sensitivity_signatures, \
    'MAX_LEN sensitivity changed the retained rule set; review before export.'
print("  -> The primary search cap is reached, but the MAX_LEN=6 rerun leaves "
      "the retained deliverable unchanged; the cap does not drive the findings.")

rule_funnel = pd.DataFrame([
    ('All rules from supported itemsets', len(rules_raw)),
    (f'Confidence ≥ {MIN_CONFIDENCE:.2f}', len(rules_confidence)),
    (f'Lift ≥ {MIN_LIFT:.2f}', len(rules)),
    ('Churn in consequent (including multi-item)', len(churn_rules)),
    ('Single churn consequent', len(churn_single_rules)),
    (f'Non-redundant: confidence gain ≥ {MIN_INCREMENTAL_CONFIDENCE:.0%} or no parent', len(nonredundant_churn_rules)),
], columns=['Funnel Stage', 'Rules'])

print('── Rule Filtering Funnel ──')
display(rule_funnel)
print(f"Rubric status: {'PASS' if len(nonredundant_churn_rules) >= 10 else 'FAIL'} "
      f"({len(nonredundant_churn_rules)}/10 non-redundant churn rules available)")
print('\n── Highest-lift non-redundant churn associations ──')
display(nonredundant_churn_rules.head(15)[
    ['antecedents', 'support', 'confidence', 'lift', 'Incremental_Confidence']
])

  MAX_LEN sensitivity: primary retained maximum=5 items at cap 5; expanded search cap=6.
  Expanded search adds 19 frequent itemsets and 1,178 raw rules.
  Retained non-redundant churn rules: 11 -> 11; expanded retained maximum=5 items.
  -> The primary search cap is reached, but the MAX_LEN=6 rerun leaves the retained deliverable unchanged; the cap does not drive the findings.
── Rule Filtering Funnel ──


,Funnel Stage,Rules
0,All rules from supported itemsets,45820
1,Confidence ≥ 0.50,6458
2,Lift ≥ 1.50,613
3,Churn in consequent (including multi-item),17
4,Single churn consequent,16
5,Non-redundant: confidence gain ≥ 1% or no parent,11


Rubric status: PASS (11/10 non-redundant churn rules available)

── Highest-lift non-redundant churn associations ──


,antecedents,support,confidence,lift,Incremental_Confidence
0,"(Age_Band_Age_46_to_60, Active_Status_Inactive...",0.0405,0.772901,3.794309,0.089292
1,"(Age_Band_Age_46_to_60, Active_Status_Inactive...",0.0313,0.726218,3.565135,0.042609
2,"(Age_Band_Age_46_to_60, Active_Status_Inactive)",0.0538,0.683609,3.355958,0.172376
3,"(Age_Band_Age_46_to_60, Geography_Germany)",0.0338,0.673307,3.305384,0.162074
4,"(Age_Band_Age_46_to_60, Gender_Female, Product...",0.0323,0.664609,3.262686,0.055564
5,"(Age_Band_Age_46_to_60, Products_Label_Product...",0.0606,0.609045,2.989913,0.097813
6,"(Age_Band_Age_46_to_60, Balance_Band_Balance_A...",0.0489,0.577332,2.834226,0.066099
7,"(Age_Band_Age_46_to_60, Gender_Female)",0.0471,0.572993,2.812924,0.061760
8,"(Active_Status_Inactive, Balance_Band_Balance_...",0.0327,0.557070,2.734756,0.036237
9,"(Active_Status_Inactive, Products_Label_Produc...",0.0375,0.520833,2.556865,0.092368


### Interpretation — frequency, strength, and novelty answer different questions

- Support prevents tiny anecdotes; confidence states how often churn accompanies the antecedent; lift compares that confidence with the full-book churn rate.
- The printed funnel distinguishes all rules mathematically generated from the much smaller decision-ready set. “Rules generated” in the appendix means the unfiltered rule universe, not the post-lift table.
- A high-lift extension is not automatically new knowledge. The incremental-confidence test compares each rule with its strongest simpler parent and removes conditions that add less than one percentage point. This makes the delivered ten rules non-trivial rather than ten cosmetic variants of the same profile.
- The age label is an explicit **46–60 dataset band**, not a demographic or legal definition of “senior.” The 100K balance split is a currency-neutral scenario threshold, not verified insurance status.

In [6]:
# ── Hypothesis Test: Germany + Inactive + 1 Product → Churn ──────────────────
print("═" * 70)
print(" HYPOTHESIS VERIFICATION: Germany + Inactive + Products_1 → Churned")
print("═" * 70)

# Find rules containing all three antecedent items
target_items = {'Geography_Germany', 'Active_Status_Inactive', 'Products_Label_Products_1'}
target_churn = frozenset(['Churn_Status_Churned'])

# Filter from the full rule set
matching_rules = churn_rules[
    churn_rules['antecedents'].apply(
        lambda x: target_items.issubset(x)
    )
]

if len(matching_rules) > 0:
    print(f"\n  ✔ Rule FOUND! ({len(matching_rules)} matching rule(s))\n")
    display(matching_rules[['antecedents','consequents',
                             'support','confidence','lift']].reset_index(drop=True))
else:
    print("\n  Rule not found at current thresholds.")
    print("  Computing metrics directly from the data...\n")

# ── Direct Computation from Raw Data ─────────────────────────────────────────
# Compute support, confidence, lift manually for verification
mask_antecedent = (
    (df['Geography'] == 'Germany') & 
    (df['IsActiveMember'] == 0) & 
    (df['NumOfProducts'] == 1)
)
mask_full = mask_antecedent & (df['Exited'] == 1)

support_ant  = mask_antecedent.mean()
support_full = mask_full.mean()
confidence   = support_full / support_ant if support_ant > 0 else 0
support_con  = df['Exited'].mean()
lift         = confidence / support_con

print(f"── Direct Calculation from Raw Data ──")
print(f"  Antecedent   (Germany ∩ Inactive ∩ 1-Product): "
      f"{mask_antecedent.sum()} records ({support_ant*100:.2f}%)")
print(f"  Itemset      (Antecedent ∩ Churned):            "
      f"{mask_full.sum()} records ({support_full*100:.2f}%)")
print(f"\n  Support:    {support_full:.4f} ({support_full*100:.2f}%)")
print(f"  Confidence: {confidence:.4f} ({confidence*100:.1f}%) ← % of 'Germany+Inactive+1prod' who churned")
print(f"  Lift:       {lift:.4f} ← {lift:.2f}x more likely to churn than baseline")
print(f"\n  BASELINE Churn Rate (full dataset): {support_con*100:.1f}%")
print(f"  SEGMENT  Churn Rate:                {confidence*100:.1f}%")


══════════════════════════════════════════════════════════════════════
 HYPOTHESIS VERIFICATION: Germany + Inactive + Products_1 → Churned
══════════════════════════════════════════════════════════════════════

  ✔ Rule FOUND! (2 matching rule(s))



,antecedents,consequents,support,confidence,lift
0,"(Active_Status_Inactive, Balance_Band_Balance_...",(Churn_Status_Churned),0.0327,0.557070,2.734756
1,"(Active_Status_Inactive, Products_Label_Produc...",(Churn_Status_Churned),0.0375,0.520833,2.556865


── Direct Calculation from Raw Data ──
  Antecedent   (Germany ∩ Inactive ∩ 1-Product): 720 records (7.20%)
  Itemset      (Antecedent ∩ Churned):            375 records (3.75%)

  Support:    0.0375 (3.75%)
  Confidence: 0.5208 (52.1%) ← % of 'Germany+Inactive+1prod' who churned
  Lift:       2.5569 ← 2.56x more likely to churn than baseline

  BASELINE Churn Rate (full dataset): 20.4%
  SEGMENT  Churn Rate:                52.1%


In [7]:
# ── Top 10 non-redundant churn rules with action commentary ─────────
top_rules = nonredundant_churn_rules.nlargest(10, 'lift')[
    ['antecedents', 'consequents', 'support', 'confidence', 'lift',
     'conviction', 'Incremental_Confidence']
].reset_index(drop=True)

def readable_item(item):
    return (item.replace('Age_Band_Age_', 'Age ')
                .replace('_to_', '–')
                .replace('Balance_Band_Balance_', 'Balance ')
                .replace('Active_Status_', '')
                .replace('Products_Label_Products_', 'Products=')
                .replace('Geography_', '')
                .replace('Gender_', '')
                .replace('CrCard_Status_', '')
                .replace('_', ' '))

def business_commentary(antecedent):
    items = set(antecedent)
    descriptors = []
    actions = []
    if 'Age_Band_Age_46_to_60' in items:
        descriptors.append('the dataset’s age 46–60 band')
    if 'Active_Status_Inactive' in items:
        descriptors.append('inactive customers')
        actions.append('test a re-engagement contact')
    if 'Products_Label_Products_1' in items:
        descriptors.append('single-product relationships')
        actions.append('test a relevant second-product offer')
    if 'Geography_Germany' in items:
        descriptors.append('the German book')
        actions.append('audit local service and product-fit drivers')
    if 'Gender_Female' in items:
        descriptors.append('female customers')
        actions.append('investigate experience differences with fairness safeguards')
    if 'Balance_Band_Balance_Above_100K' in items:
        descriptors.append('balances above the 100K scenario cut')
        actions.append('prioritize relationship-manager review while sensitivity-testing the threshold')
    if 'CrCard_Status_Has_CrCard' in items:
        descriptors.append('card holders')
    profile_text = ', '.join(descriptors) if descriptors else 'this measured profile'
    action_text = '; '.join(dict.fromkeys(actions)) or 'validate the segment in a later period before action'
    return f'Association concentrates among {profile_text}. Next step: {action_text}; do not treat the rule as causal.'

top_rules['IF (Conditions)'] = top_rules['antecedents'].apply(
    lambda items: ' ∩ '.join(sorted(readable_item(item) for item in items))
)
top_rules['THEN (Outcome)'] = 'Churned'
top_rules['Support (%)'] = (top_rules['support'] * 100).round(2)
top_rules['Confidence (%)'] = (top_rules['confidence'] * 100).round(1)
top_rules['Lift'] = top_rules['lift'].round(3)
top_rules['Conviction'] = top_rules['conviction'].round(3)
top_rules['Confidence gain vs best parent (pp)'] = (
    top_rules['Incremental_Confidence'] * 100
).round(1)
top_rules['Business Commentary'] = top_rules['antecedents'].apply(business_commentary)

# Individually authored headline commentary, keyed by antecedent so a rerun
# cannot attach an interpretation to the wrong rule. The generator above is
# retained as the complete fallback for every other documented rule.
MANUAL_COMMENTARY = {
    frozenset({'Age_Band_Age_46_to_60', 'Active_Status_Inactive',
               'Products_Label_Products_1'}):
        "The strongest compound association has 77.3% confidence and 3.794 "
        "lift, adding 8.9 percentage points over its best simpler parent. "
        "The actionable levers are inactivity and a one-product relationship, "
        "not age itself: test re-engagement plus a relevant second-product "
        "offer against a holdout before any wider rollout.",
    frozenset({'Geography_Germany', 'Active_Status_Inactive',
               'Products_Label_Products_1'}):
        "The assigned Germany hypothesis is supported: inactive, one-product "
        "German customers show 52.1% confidence and 2.557 lift, a 9.2-point "
        "gain over the best simpler parent. Geography may proxy for market or "
        "service differences, so audit the German service journey and product "
        "fit before attributing the pattern to customers.",
    frozenset({'Age_Band_Age_46_to_60', 'Gender_Female',
               'Products_Label_Products_1'}):
        "Among women aged 46–60 with one product, churn confidence is 66.5% "
        "with 3.263 lift and a 5.6-point gain over the best simpler parent. "
        "Use this as a prompt for safeguarded experience research and a "
        "controlled product-depth test—not for gender-based targeting.",
}

top_antecedent_keys = set(top_rules['antecedents'].map(frozenset))
missing_manual_keys = set(MANUAL_COMMENTARY) - top_antecedent_keys
assert not missing_manual_keys, \
    f'Manual commentary keys no longer appear in the top rules: {missing_manual_keys}'

def final_commentary(antecedent):
    return MANUAL_COMMENTARY.get(
        frozenset(antecedent), business_commentary(antecedent)
    )

top_rules['Business Commentary'] = top_rules['antecedents'].apply(final_commentary)

display_cols = [
    'IF (Conditions)', 'THEN (Outcome)', 'Support (%)', 'Confidence (%)',
    'Lift', 'Confidence gain vs best parent (pp)', 'Business Commentary'
]
print('── TOP 10 ASSOCIATION RULES — NON-REDUNDANT CHURN PROFILE DISCOVERY ──')
display(top_rules[display_cols])

top_rule_export_cols = [
    'antecedents', 'consequents', 'support', 'confidence',
    'IF (Conditions)', 'THEN (Outcome)', 'Support (%)', 'Confidence (%)',
    'Lift', 'Conviction', 'Confidence gain vs best parent (pp)',
    'Business Commentary',
]
top_rules[top_rule_export_cols].to_csv(TOP_RULES_PATH, index=False)
rules.to_csv(ALL_RULES_PATH, index=False)

phase3_metric_rows = [
    {'Phase': 'Phase 3', 'Metric': 'Rules generated', 'Value': len(rules_raw), 'Unit': 'rules',
     'Definition': 'All association rules generated from frequent itemsets before confidence/lift filters.', 'Source': 'phase3_association_rules.ipynb'},
    {'Phase': 'Phase 3', 'Metric': 'Rules after confidence filter', 'Value': len(rules_confidence), 'Unit': 'rules',
     'Definition': f'Rules with confidence ≥ {MIN_CONFIDENCE:.2f}.', 'Source': 'phase3_association_rules.ipynb'},
    {'Phase': 'Phase 3', 'Metric': 'Rules after confidence and lift filters', 'Value': len(rules), 'Unit': 'rules',
     'Definition': f'Rules with confidence ≥ {MIN_CONFIDENCE:.2f} and lift ≥ {MIN_LIFT:.2f}, after leakage guards.', 'Source': 'phase3_association_rules.ipynb'},
    {'Phase': 'Phase 3', 'Metric': 'Non-redundant churn rules retained', 'Value': len(nonredundant_churn_rules), 'Unit': 'rules',
     'Definition': f'Single-churn-consequent rules adding ≥{MIN_INCREMENTAL_CONFIDENCE:.0%} confidence over the best proper subset, or having no parent.', 'Source': 'phase3_association_rules.ipynb'},
    {'Phase': 'Phase 3', 'Metric': 'Documented churn rules', 'Value': len(top_rules), 'Unit': 'rules',
     'Definition': 'Ranked non-redundant rules with individual business commentary.', 'Source': 'phase3_association_rules.ipynb'},
    {'Phase': 'Phase 3', 'Metric': 'Highest lift (retained churn rules)', 'Value': float(nonredundant_churn_rules['lift'].max()), 'Unit': 'lift',
     'Definition': 'Maximum lift among the non-redundant churn-rule set.', 'Source': 'phase3_association_rules.ipynb'},
]
phase3_metrics = upsert_evaluation_metrics(phase3_metric_rows)
print('\nRules and Phase 3 metrics saved to outputs/.')
display(phase3_metrics[phase3_metrics['Phase'] == 'Phase 3'])

print(f'''
KEY DISCOVERY
The strongest retained association has lift {top_rules.loc[0, 'Lift']:.3f} and
confidence {top_rules.loc[0, 'Confidence (%)']:.1f}%. Its value is the
interaction among conditions—not any single field in isolation. All ten rows
remain descriptive hypotheses for validation and controlled testing, not a
churn-prediction score.''')

── TOP 10 ASSOCIATION RULES — NON-REDUNDANT CHURN PROFILE DISCOVERY ──


,IF (Conditions),THEN (Outcome),Support (%),Confidence (%),Lift,Confidence gain vs best parent (pp),Business Commentary
0,Age 46–60 ∩ Inactive ∩ Products=1,Churned,4.05,77.3,3.794,8.9,The strongest compound association has 77.3% c...
1,Age 46–60 ∩ Balance Above 100K ∩ Inactive,Churned,3.13,72.6,3.565,4.3,Association concentrates among the dataset’s a...
2,Age 46–60 ∩ Inactive,Churned,5.38,68.4,3.356,17.2,Association concentrates among the dataset’s a...
3,Age 46–60 ∩ Germany,Churned,3.38,67.3,3.305,16.2,Association concentrates among the dataset’s a...
4,Age 46–60 ∩ Female ∩ Products=1,Churned,3.23,66.5,3.263,5.6,"Among women aged 46–60 with one product, churn..."
5,Age 46–60 ∩ Products=1,Churned,6.06,60.9,2.990,9.8,Association concentrates among the dataset’s a...
6,Age 46–60 ∩ Balance Above 100K,Churned,4.89,57.7,2.834,6.6,Association concentrates among the dataset’s a...
7,Age 46–60 ∩ Female,Churned,4.71,57.3,2.813,6.2,Association concentrates among the dataset’s a...
8,Balance Above 100K ∩ Germany ∩ Inactive ∩ Prod...,Churned,3.27,55.7,2.735,3.6,Association concentrates among inactive custom...
9,Germany ∩ Inactive ∩ Products=1,Churned,3.75,52.1,2.557,9.2,The assigned Germany hypothesis is supported: ...



Rules and Phase 3 metrics saved to outputs/.


,Phase,Metric,Value,Unit,Definition,Source
13,Phase 3,Documented churn rules,10.000000,rules,Ranked non-redundant rules with individual bus...,phase3_association_rules.ipynb
14,Phase 3,Highest lift (retained churn rules),3.794309,lift,Maximum lift among the non-redundant churn-rul...,phase3_association_rules.ipynb
15,Phase 3,Non-redundant churn rules retained,11.000000,rules,Single-churn-consequent rules adding ≥1% confi...,phase3_association_rules.ipynb
16,Phase 3,Rules after confidence and lift filters,613.000000,rules,"Rules with confidence ≥ 0.50 and lift ≥ 1.50, ...",phase3_association_rules.ipynb
17,Phase 3,Rules after confidence filter,6458.000000,rules,Rules with confidence ≥ 0.50.,phase3_association_rules.ipynb
18,Phase 3,Rules generated,45820.000000,rules,All association rules generated from frequent ...,phase3_association_rules.ipynb



KEY DISCOVERY
The strongest retained association has lift 3.794 and
confidence 77.3%. Its value is the
interaction among conditions—not any single field in isolation. All ten rows
remain descriptive hypotheses for validation and controlled testing, not a
churn-prediction score.
